# Reproducing the IBL brain-wide map: results guide and first experiment

Target: **A brain-wide map of neural activity during complex behaviour** (IBL et al., Nature, 2025),
[doi:10.1038/s41586-025-09235-0](https://doi.org/10.1038/s41586-025-09235-0).

This notebook maps results to code and runs a small stimulus-selectivity experiment from one probe.
The example exercises the published analysis method; it does **not** reproduce a brain-wide figure
or estimate the fraction of selective neurons across the brain.

From the repository root, install and launch with:

```sh
uv sync --locked --group notebook
uv run --locked --group notebook jupyter lab notebooks/reproduce_bwm.ipynb
```

Select the project's `.venv/bin/python` kernel. The first data run downloads a processed probe/session
and may take several minutes; it does not download raw electrophysiology or videos. The statistical
step uses five units and the existing 3,000-shuffle procedure. Downloads are reused through ONE's cache.
See the [reproduction guide](../docs/reproduction.md) for dependency groups, blockers, and validation.


## 1. Code structure and data flow

```text
brainwidemap/bwm_loading.py       frozen queries, QC, spikes, trials, aggregates
    ├── single_cell_stats/       condition-combined shuffle tests
    ├── decoding/                neural activity → task/behavior variables
    ├── encoding/                events/behavior → single-neuron activity
    └── manifold/                PETHs → population trajectories and distances
                  ↓
brainwidemap/meta/               pooled comparisons and paper figure assembly
```

- **eid** identifies a recording session; **pid** identifies a probe insertion within it.
- `load_good_units` filters units and reindexes clusters. `spikes['clusters']` indexes the returned
  cluster table, while its `cluster_id` column records the original sorter ID. `uuids` identifies units.
- `load_trials_and_mask` returns trials and a Boolean inclusion mask. The example follows the
  single-cell wrapper's 80 ms–2 s reaction-time window, required-event checks, and performance truncation.
- `fixtures/2023_12_bwm_release.csv` fixes the released session/probe cohort. This cohort freeze is
  distinct from aggregate-table tags and individual dataset revisions.
- Decoding uses numbered caching/SLURM scripts; encoding uses `pipelines/01` through `04`.
  Figure scripts expect their outputs in specific directories; importing some scripts starts downloads.


## 2. Results production checklist

These are starting points for reproduction, not a claim that every figure can run unchanged.
Main-figure labels 4–7 and the ED/supplementary labels below are explicit in the figure source.
Figures 1–3 have thematic entrypoints; their exact panel-to-code mapping still needs checking against
source data and the final paper. The full atlas figures need pooled results from many sessions.

| Paper result | Code entrypoint (relative to repo root) | Inputs / groups | Output and scale | Status here |
|---|---|---|---|---|
| Figs. 1–2: task, example recordings, dataset/coverage | `meta/intro_figure_example_time_series.py:bwm_data_series_fig`; `meta/inclusion_info.py`; `bwm_loading.py:bwm_query` (all under `brainwidemap/`) | Frozen cohort; probe data or historical aggregates; base | Counts, coverage and recording panels; cohort-wide for coverage | Frozen cohort counts run below; full panel mapping partial |
| Fig. 3: analysis-method illustrations | `decoding/functions/decoding.py:fit_eid`; `single_cell_stats/*`; `manifold/state_space_bwm.py:get_d_vars`; `encoding/design.py:generate_design` | One-session data; respective analysis groups | Decoding, shuffle statistics, trajectories, GLM examples | Stimulus single-cell method runs below; exact illustration panels unverified |
| Fig. 4: visual stimulus | `meta/bwm_figs.py:main_fig_for_variable('stim')` | Four pooled analyses; all analysis groups plus external figure tools | Brain-wide stimulus figure; full cohort | Five-unit stimulus example only |
| Fig. 5: choice | `meta/bwm_figs.py:main_fig_for_variable('choice')` | Four pooled analyses and example fits | Brain-wide choice figure; full cohort | Cataloged |
| Fig. 6: feedback | `meta/bwm_figs.py:main_fig_for_variable('fback')` | Four pooled analyses and example fits | Brain-wide feedback figure; full cohort | Cataloged |
| Fig. 7: wheel movement | `meta/bwm_figs.py:main_fig_for_wheel` | Wheel regressors, decoding/encoding outputs | Speed/velocity figure; full cohort | Cataloged |
| ED Figs. 2–3: comparisons across analyses | `meta/bwm_figs.py:plot_eds_figures`; `plot_swansons_across_analysis`; `plot_scatter_analysis_pairs` | Pooled regional scores | Flatmaps and pairwise comparisons | Cataloged; external figure prerequisites |
| ED Figs. 6, 11, 14: stimulus/choice/feedback regional views | `meta/bwm_figs.py:plot_eds_figures` | Decoding summaries, atlas | Regional flatmaps and slices | Labels confirmed in code; cataloged |
| ED Fig. 9: encoding / reaction-time splits | `meta/bwm_figs.py:plot_eds_figures` | Actual early/late fits must replace placeholders | Encoding panels | Source explicitly warns of placeholder data; not reproduction-ready |
| Supplementary Fig. 3: sampling | `meta/bwm_figs.py:plot_supp_figures`; `manifold/state_space_bwm.py:plot_strip_sampling` | Population results; manifold group | Neuron/session subsampling comparisons | Cataloged |
| Supplementary Figs. 4, 6–8 | `meta/bwm_figs.py:plot_supp_figures` | Sex/reproducibility, feedback-baseline, speed/velocity, and training data | Control figures | Labels in code; cataloged, not validated against final supplement |
| Other extended/supplementary results | `meta/`, `encoding/scripts/`, `manifold/state_space_bwm.py` | Result-specific data and dependencies | Additional controls | Inventory incomplete; check final supplement before claiming coverage |

Paths in the table omit the common `brainwidemap/` prefix unless stated otherwise.
The [source figure/text repository](https://github.com/int-brain-lab/Brainwide-Map-Paper) is a separate
upstream resource. Companion-paper reproduction is outside this first notebook.


## 3. Configure a direct connection and local output paths

Run this cell **before importing ONE or other networking libraries**. Proxy settings are disabled for
this Python process, including libraries such as requests and boto3. This does not change your shell.
`BWM_CACHE_DIR` and `BWM_RESULTS_DIR` can point to larger disks. The defaults are ignored by Git.


In [ ]:
import os

for key in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
    os.environ.pop(key, None)
os.environ["NO_PROXY"] = "*"
os.environ["no_proxy"] = "*"

from pathlib import Path

ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "brainwidemap" / "fixtures").is_dir() and (p / "pyproject.toml").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or notebooks/ directory.")
CACHE_DIR = Path(os.environ.get("BWM_CACHE_DIR", str(ROOT / "data" / "one"))).expanduser().resolve()
RESULTS_DIR = Path(os.environ.get("BWM_RESULTS_DIR", str(ROOT / "results" / "single_probe"))).expanduser().resolve()
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
print(f"Repository: {ROOT}\nONE cache: {CACHE_DIR}\nResults: {RESULTS_DIR}")


In [ ]:
import hashlib
import importlib.metadata as metadata
import json
import platform
import subprocess
import sys
import time
from datetime import datetime, timezone

from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from one.api import ONE
from brainbox.io.one import SessionLoader
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.singlecell import bin_spikes2D
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask
from brainwidemap.bwm_loading import MODIFIED_BEFORE
from brainwidemap.single_cell_stats.single_cell_Working_example_stimulus import get_stim_time_shuffle

run_started = time.perf_counter()
FREEZE = "2023_12_bwm_release"
EID = "15f742e1-1043-45c9-9504-f1e8a53c1744"
PID = "3675290c-8134-4598-b924-83edb7940269"
N_UNITS = 5
N_SHUFFLES = 3000
SEED = 0
TIME_WINDOW = (0.0, 0.1)
BASE_URL = "https://openalyx.internationalbrainlab.org"
print(f"Python {sys.version.split()[0]} at {sys.executable}")
print({name: metadata.version(name) for name in ("brainwidemap", "ibllib", "ONE-api", "numpy", "scipy")})


## 4. Verify the frozen cohort (no data download)

This check reads the packaged CSV, not a live database query. It counts recordings before the
analysis-specific unit/region filters. Do not confuse these counts with the paper's final canonical
neuron and region counts.


In [ ]:
bwm_df = bwm_query(freeze=FREEZE)
cohort_counts = {
    "insertions": len(bwm_df),
    "sessions": bwm_df.eid.nunique(),
    "subjects": bwm_df.subject.nunique(),
}
assert cohort_counts == {"insertions": 699, "sessions": 459, "subjects": 139}, cohort_counts
example = bwm_df.loc[bwm_df.pid == PID]
assert len(example) == 1 and example.iloc[0].eid == EID
PROBE_NAME = example.iloc[0].probe_name
display(pd.Series(cohort_counts, name=FREEZE).to_frame())
display(example)


## 5. Fetch one probe and apply the existing QC

The selected session/probe also appears in `test_single_cell_stats.py`. ONE connects to the public
OpenAlyx endpoint and caches processed spikes, cluster information, and trials. Its initial connection
can also create small ONE connection-settings files in your user configuration directory.
Set `BWM_ONE_TOKEN` before launching Jupyter to use your own API token instead of the shared
public demo login. ONE currently warns that the shared account will be retired.

We retain the loader's spike-sorting revision (`2024-05-06`) and trial cutoff (`MODIFIED_BEFORE`).
This example does not call `bwm_units()` or download aggregate tables: their default release tag is
currently newer than the historical paper aggregates, and the new tag lacks a trials checksum.


In [ ]:
# A personal API token can replace the shared public demo login; never print or save the token.
auth = ({"token": os.environ["BWM_ONE_TOKEN"]} if os.environ.get("BWM_ONE_TOKEN")
        else {"username": "intbrainlab", "password": "international"})
one = ONE(base_url=BASE_URL, silent=True, cache_dir=CACHE_DIR, **auth)
load_started = time.perf_counter()
spikes, clusters = load_good_units(
    one, PID, compute_metrics=True, eid=EID, pname=PROBE_NAME,
)
session_loader = SessionLoader(one=one, eid=EID, revision=MODIFIED_BEFORE)
session_loader.load_trials()
n_trials_loaded = len(session_loader.trials)
trials, mask = load_trials_and_mask(
    one, EID, min_rt=0.08, max_rt=2.0, nan_exclude="default", sess_loader=session_loader,
)
good_trials = trials.loc[mask].copy()
load_seconds = time.perf_counter() - load_started

assert len(spikes["times"]) > 0 and len(good_trials) > 0
assert len({len(v) for v in spikes.values()}) == 1
assert np.all(np.diff(spikes["times"]) >= 0)
assert np.isin(spikes["clusters"], clusters.index).all()
assert (clusters["label"] >= 1.0).all()
assert len(mask) == len(trials) and not mask.isna().any()
required_events = ["stimOn_times", "choice", "feedback_times", "probabilityLeft", "firstMovement_times", "feedbackType"]
assert good_trials[required_events].notna().all().all()
rt = good_trials.firstMovement_times - good_trials.stimOn_times
assert rt.between(0.08, 2.0).all()

qc_summary = {
    "loaded_trials": n_trials_loaded,
    "performance_truncated_trials": n_trials_loaded - len(trials),
    "masked_out_trials_after_truncation": int((~mask).sum()),
    "included_trials": len(good_trials),
    "good_units": len(clusters),
    "spikes": len(spikes["times"]),
}
display(pd.Series(qc_summary, name="count").to_frame())
print(f"Loading and QC: {load_seconds:.1f} s")


## 6. Select units before testing

The first five eligible units in **original sorter cluster-ID order** are selected, without looking
at responses or p-values. No brain-wide region-count filter is applied to this single-probe example.
Counts use the same 0–100 ms post-stimulus interval as `BWM_stim_test`.


In [ ]:
stim_on = good_trials.stimOn_times.to_numpy()
intervals = np.column_stack((stim_on + TIME_WINDOW[0], stim_on + TIME_WINDOW[1]))
spike_counts, count_cluster_ids = get_spike_counts_in_bins(
    spikes["times"], spikes["clusters"], intervals,
)
eligible_units = clusters.loc[count_cluster_ids].sort_values("cluster_id").head(N_UNITS).copy()
assert len(eligible_units) == N_UNITS, "This probe has fewer eligible units than requested."
selected_cluster_ids = eligible_units.index.to_numpy()
count_rows = {int(cluster_id): i for i, cluster_id in enumerate(count_cluster_ids)}
selected_rows = [count_rows[int(cluster_id)] for cluster_id in selected_cluster_ids]
rates = spike_counts[selected_rows] / (TIME_WINDOW[1] - TIME_WINDOW[0])

contrast_left = good_trials.contrastLeft.to_numpy()
contrast_right = good_trials.contrastRight.to_numpy()
choices = good_trials.choice.to_numpy()
blocks = good_trials.probabilityLeft.to_numpy()
conditions = []
for block in (0.8, 0.2):
    for choice in (1, -1):
        matched = (blocks == block) & (choices == choice)
        conditions.append({
            "probabilityLeft": block, "choice": choice,
            "left_trials": int(np.sum(matched & (contrast_left > 0))),
            "right_trials": int(np.sum(matched & (contrast_right > 0))),
        })
condition_counts = pd.DataFrame(conditions)
assert (condition_counts.left_trials * condition_counts.right_trials).sum() > 0
unit_columns = [c for c in ("cluster_id", "uuids", "acronym", "atlas_id", "label") if c in eligible_units]
display(eligible_units[unit_columns])
display(condition_counts)


## 7. Existing condition-combined stimulus shuffle test

This calls `get_stim_time_shuffle`, the same helper used by `BWM_stim_test`, on the selected rows.
It combines stimulus comparisons across choice/block conditions and shuffles within temporal blocks.
The number of shuffles is unchanged from the wrapper; the number of units is reduced.

The fixed seed makes this run repeatable with the locked environment and input revisions. P-values
are reported without a brain-wide multiple-testing correction; the example is not a regional
significance result. Expect this step to take longer than plotting.


In [ ]:
np.random.seed(SEED)
stats_started = time.perf_counter()
p_values = get_stim_time_shuffle(
    rates, contrast_left, contrast_right, blocks, choices, nShuf=N_SHUFFLES,
)
stats_seconds = time.perf_counter() - stats_started
assert p_values.shape == (N_UNITS,)
assert np.isfinite(p_values).all() and ((p_values >= 0) & (p_values <= 1)).all()

results = eligible_units[unit_columns].copy()
results.insert(0, "reindexed_cluster_id", selected_cluster_ids)
results.insert(0, "pid", PID)
results.insert(0, "eid", EID)
results["stimulus_p_value"] = p_values
results["n_shuffles"] = N_SHUFFLES
results["seed"] = SEED
results.to_csv(RESULTS_DIR / "stimulus_selectivity.csv", index=False)
display(results)
print(f"Shuffle test: {stats_seconds:.1f} s for {N_UNITS} units, {N_SHUFFLES} shuffles")


## 8. Stimulus-aligned firing rates

These descriptive PETHs pool positive-contrast left and right trials separately, with mean ± SEM.
Unlike the statistical test, this plot does not condition on choice and block; it helps inspect the
selected cells rather than isolate stimulus coding. Unit selection was completed before testing.


In [ ]:
# bin_spikes2D expects spike arrays restricted to the requested clusters, in sorted ID order.
peth_cluster_ids = np.sort(selected_cluster_ids)
peth_spike_mask = np.isin(spikes["clusters"], peth_cluster_ids)
binned, bin_times = bin_spikes2D(
    spikes["times"][peth_spike_mask], spikes["clusters"][peth_spike_mask],
    peth_cluster_ids, stim_on, pre_time=0.2, post_time=0.4, bin_size=0.01,
)
peth_rows = [int(np.flatnonzero(peth_cluster_ids == c)[0]) for c in selected_cluster_ids]
binned_hz = binned[:, peth_rows, :] / 0.01
fig, axes = plt.subplots(N_UNITS, 1, figsize=(8, 2.2 * N_UNITS), sharex=True, squeeze=False)
for i, ax in enumerate(axes[:, 0]):
    for label, trial_selection, color in (
        ("Left stimulus", contrast_left > 0, "#2369bd"),
        ("Right stimulus", contrast_right > 0, "#a8373b"),
    ):
        values = binned_hz[trial_selection, i, :]
        assert len(values) > 1, f"Not enough {label.lower()} trials for a PETH."
        mean = values.mean(axis=0)
        sem = values.std(axis=0, ddof=1) / np.sqrt(len(values))
        ax.plot(bin_times, mean, label=f"{label} (n={len(values)})", color=color)
        ax.fill_between(bin_times, mean - sem, mean + sem, color=color, alpha=0.2)
    ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
    unit = eligible_units.iloc[i]
    region = unit.get("acronym", str(unit.atlas_id))
    ax.set_title(f"Unit {int(unit.cluster_id)} · {region} · shuffle p={p_values[i]:.4g}")
    ax.set_ylabel("Rate (Hz)")
axes[0, 0].legend(frameon=False)
axes[-1, 0].set_xlabel("Time from stimulus onset (s)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "stimulus_peth.png", dpi=150)
fig.savefig(RESULTS_DIR / "stimulus_peth.pdf")
plt.show()


## 9. Save provenance and the next reproduction steps

The JSON records package versions, cohort/loader revisions, unit IDs, parameters, source hashes,
and timings. The executed notebook and ONE's cache provide additional context. Input files can be
patched upstream, so a fixed seed alone is not a guarantee of historical paper identity.


In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

revision = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True, capture_output=True, check=False,
).stdout.strip()
provenance = {
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "git_revision": revision,
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {d.metadata["Name"]: d.version for d in metadata.distributions() if d.metadata["Name"]},
    "source_sha256": {
        str(path.relative_to(ROOT)): sha256(path)
        for path in (
            ROOT / "uv.lock",
            ROOT / "notebooks" / "reproduce_bwm.ipynb",
            ROOT / "brainwidemap" / "bwm_loading.py",
            ROOT / "brainwidemap" / "single_cell_stats" / "single_cell_Working_example_stimulus.py",
            ROOT / "brainwidemap" / "single_cell_stats" / "single_cell_util.py",
        )
    },
    "base_url": BASE_URL, "freeze": FREEZE, "eid": EID, "pid": PID, "probe": PROBE_NAME,
    "spike_sorting_revision": "2024-05-06", "trial_revision": MODIFIED_BEFORE,
    "qc": qc_summary,
    "parameters": {"n_units": N_UNITS, "n_shuffles": N_SHUFFLES, "seed": SEED, "window_s": TIME_WINDOW},
    "selected_sorter_cluster_ids": eligible_units.cluster_id.astype(int).tolist(),
    "stimulus_p_values": p_values.tolist(),
    "timings_s": {"load_and_qc": load_seconds, "statistics": stats_seconds, "total": time.perf_counter() - run_started},
    "cache_dir": str(CACHE_DIR), "results_dir": str(RESULTS_DIR),
}
condition_counts.to_csv(RESULTS_DIR / "trial_conditions.csv", index=False)
(RESULTS_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")
print(f"Saved results in {RESULTS_DIR}")
print(json.dumps(provenance["timings_s"], indent=2))


### Continue toward paper-wide results

1. **Historical cohort/unit counts:** resolve aggregate release selection before using `bwm_units()`;
   see the release caveat in the guide. `2024_Q2_IBL_et_al_BWM` has checksums for trials and clusters,
   whereas the current default `2026_Q2_IBL_et_al_BWM` does not include trials in its checksum manifest.
   Do not silently mix these releases.
2. **Single-cell statistics:** use the `BWM_*_test` wrappers for stimulus, choice, feedback, and block
   tests across insertions, retaining QC and paper-specific region and multiplicity rules.
3. **Decoding:** enable the decoding group; read the [pipeline README](../brainwidemap/decoding/README.md),
   configure `settings.py`, cache sessions/null targets, fit sessions, and generate summary tables.
4. **Encoding:** enable the encoding group; start with the
   [existing fitting notebook](../brainwidemap/encoding/scripts/fit_example_session.ipynb), then configure
   cache/output paths and the four pipeline stages. It is a legacy example, not validated by this run.
5. **Population trajectories:** enable the manifold group; inspect import-time data loading first,
   then compute per-insertion PETHs, stack results, and plot region-level trajectories/distances.
6. **Publication figures:** locate/compute the required pooled result files and supply external figure
   tools. Review local paths, the forced Qt backend, missing `GLMPredictor` import, and explicit
   placeholder panels before running `meta/bwm_figs.py`.

The installation and this example are the first reproducible slice. Full-cohort figures and
SLURM/legacy-container execution remain separate validation tasks.
